# KAUST Project 9 — ZeroWaste-f External-Domain Test (Phase 1)

**Purpose:** Evaluate the **already trained TrashNet classifier** on ground-truth object crops from ZeroWaste-f (**no retraining**). The output is measured accuracy and macro-F1 on overlapping material classes (cardboard, metal, plastic).

**Important:** The official ZeroWaste-f archive is approximately **7.5 GB**; downloading may take time and free Colab storage. Original dataset: https://zenodo.org/records/6412647. The dataset is **CC BY-NC 4.0**, for non-commercial use; don't upload images to GitHub. This notebook requires the original archive to compute real results.

**You need:** `best_model.pt` (from the Phase 1 results) and ZeroWaste-f. We cannot compute real ZeroWaste numbers without the images and their COCO annotations.

In [ ]:
!pip -q install pyzipper scipy scikit-learn
import os, pathlib
from google.colab import files
print('Upload best_model.pt (NOT an API key).')
uploaded = files.upload()
pts=[n for n in uploaded if n.lower().endswith('.pt')]
assert len(pts)==1,'Please upload exactly one best_model.pt checkpoint'
CKPT=pts[0]
print('Model loaded:',CKPT)

## Dataset source

By default the notebook downloads the **official full archive** directly to Colab and uses only a small annotated sample. If you've already placed the archive in Colab, set `ARCHIVE` to that path and skip downloading. **No Google Drive authorization needed**.

*Zenodo documents an extraction password, which the evaluation script handles.*

In [ ]:
from pathlib import Path
ARCHIVE=Path('/content/zerowaste-f-final.zip')
if not ARCHIVE.exists():
    import subprocess
    url='https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
    print('Downloading official 7.5GB ZeroWaste-f archive; monitor Colab disk space.')
    subprocess.run(['wget','-c','-O',str(ARCHIVE),url],check=True)
assert ARCHIVE.stat().st_size > 1024*1024*100, 'Archive is unexpectedly small. Check download errors.'
print('Downloaded / located:',ARCHIVE, 'size GB:',round(ARCHIVE.stat().st_size/1e9,2))

## Evaluate without retraining

We select at most 25 *different source frames* per compatible material class, from the annotated validation split. ZeroWaste soft and rigid plastics are mapped to the existing TrashNet `plastic` label. This is a **limited domain-shift check**, not detector mAP. Report exact sample size.

In [ ]:
from pathlib import Path
SCRIPT = Path('/content/zerowaste_external_eval.py')
SCRIPT.write_text('#!/usr/bin/env python3\n"""ZeroWaste-f external-domain evaluation of the existing TrashNet ResNet18.\nThis script DOES NOT train or alter the checkpoint. Requires original licensed ZeroWaste-f archive.\nGround truth is taken strictly from the COCO annotations in the archive.\nResults are only for labels matching TrashNet: cardboard, metal, and plastic (soft/rigid combined).\n"""\nimport argparse, json, random, re, csv, io, math, os\nfrom pathlib import Path, PurePosixPath\nfrom collections import defaultdict\nfrom PIL import Image\nimport numpy as np\nimport torch\nfrom torchvision import models, transforms\nfrom sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix\n\nMATERIALS={"cardboard":"cardboard","metal":"metal","soft_plastic":"plastic","rigid_plastic":"plastic", "softplastic":"plastic","rigidplastic":"plastic"}\nPASSWORD=b\'UP#1VuX409z4\'  # documented at Zenodo records/6412647\n\ndef norm(s):\n    return re.sub(r\'[^a-z0-9]+\',\'_\',str(s).lower()).strip(\'_\')\n\ndef open_archive(path):\n    import zipfile\n    try:\n        import pyzipper\n        z=pyzipper.AESZipFile(path)\n    except ImportError:\n        z=zipfile.ZipFile(path)\n    z.setpassword(PASSWORD)\n    return z\n\ndef read_member(z, n):\n    try:return z.read(n,pwd=PASSWORD)\n    except (RuntimeError, TypeError):return z.read(n)\n\ndef find_ann(names, split):\n    options=[n for n in names if n.endswith(\'.json\') and PurePosixPath(n).name.lower() in (\'labels.json\',\'instances.json\',\'annotations.json\')]\n    matched=[n for n in options if any(part.lower()==split for part in PurePosixPath(n).parts)]\n    if matched:return min(matched,key=len)\n    raise ValueError(f\'No {split}/labels.json found in archive. Available candidates: {options[:25]}\')\n\ndef get_img_member(img_name, ann_path, names, by_basename):\n    # Many releases keep images under <split>/data, while annotations sit at <split>/labels.json\n    p=PurePosixPath(img_name)\n    base=PurePosixPath(ann_path).parent\n    for candidate in (str(p), str(base/p), str(base/\'data\'/p),str(base/\'images\'/p)):\n        if candidate in names:return candidate\n    cands=by_basename.get(p.name,[])\n    prefix=\'/\'.join(base.parts)+\'/\'\n    valid=[n for n in cands if n.startswith(prefix)]\n    if valid:return valid[0]\n    return cands[0] if len(cands)==1 else None\n\ndef parse_args():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--archive\',required=True)\n    p.add_argument(\'--checkpoint\',required=True)\n    p.add_argument(\'--output\',default=\'zw_eval_outputs\')\n    p.add_argument(\'--split\',default=\'val\',choices=(\'val\',\'test\'))\n    p.add_argument(\'--per-class\',type=int,default=25)\n    p.add_argument(\'--min-side\',type=int,default=32)\n    p.add_argument(\'--seed\',type=int,default=42)\n    return p.parse_args()\n\ndef main():\n    args=parse_args(); rng=random.Random(args.seed)\n    torch.manual_seed(args.seed); np.random.seed(args.seed)\n    dst=Path(args.output);dst.mkdir(parents=True,exist_ok=True)\n    ck=torch.load(args.checkpoint,map_location=\'cpu\',weights_only=False)\n    class_names=ck.get(\'classes\')\n    if class_names is None or ck.get(\'kind\') != \'finetune\':\n        raise ValueError(\'Expected best_model.pt for ResNet18 Fine-Tuning with classes\')\n    img_size=int(ck.get(\'img_size\',160))\n    net=models.resnet18(weights=None);net.fc=torch.nn.Linear(net.fc.in_features,len(class_names))\n    net.load_state_dict(ck[\'state_dict\'],strict=True);net.eval()\n    tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])\n    with open_archive(args.archive) as z:\n        members=z.namelist();member_set=set(members)\n        by_basename=defaultdict(list)\n        for n in members:by_basename[PurePosixPath(n).name].append(n)\n        ann_file=find_ann(members,args.split)\n        ann=json.loads(read_member(z,ann_file))\n        cats={int(d[\'id\']):norm(d[\'name\']) for d in ann[\'categories\']}\n        img_rows={int(d[\'id\']):d for d in ann[\'images\']}\n        groups=defaultdict(list)\n        for a in ann[\'annotations\']:\n            ground=MATERIALS.get(cats.get(int(a[\'category_id\']),\'\'))\n            if ground is None:continue\n            box=a.get(\'bbox\') or []\n            if len(box)!=4 or min(float(box[2]),float(box[3]))<args.min_side:continue\n            img=img_rows.get(int(a[\'image_id\']))\n            if img is None:continue\n            groups[ground].append((img,a,cats.get(int(a[\'category_id\']),\'\')))\n        if not groups:raise ValueError(\'No compatible category annotations found. Dataset categories: \'+str(cats))\n        selection=[];used=set()\n        for label, candidates in sorted(groups.items()):\n            # Only one box per frame, to avoid artificially inflated sample size.\n            rng.shuffle(candidates)\n            count=0\n            for img,a,original in candidates:\n                if int(a[\'image_id\']) in used:continue\n                used.add(int(a[\'image_id\']))\n                selection.append((img,a,label,original));count+=1\n                if count>=args.per_class:break\n        rows=[];missing=0\n        for img,a,ground,original in selection:\n            member=get_img_member(img[\'file_name\'],ann_file,member_set,by_basename)\n            if member is None:missing+=1;continue\n            try:\n                raw=read_member(z,member)\n                im=Image.open(io.BytesIO(raw)).convert(\'RGB\')\n                x,y,w,h=[float(v) for v in a[\'bbox\']]\n                crop=im.crop((max(0,int(x)),max(0,int(y)),min(im.width,int(x+w)),min(im.height,int(y+h))))\n                if min(crop.size)<2:continue\n                inp=tf(crop).unsqueeze(0)\n                with torch.inference_mode():\n                    probs=net(inp).softmax(dim=1)[0]\n                pred=int(probs.argmax())\n                rows.append(dict(image_id=a[\'image_id\'],annotation_id=a[\'id\'],gt_zerowaste=original,\n                   gt_trashnet=ground,prediction=class_names[pred],confidence=round(float(probs[pred]),6),\n                   correct=(class_names[pred]==ground),split=args.split))\n            except Exception as e:\n                print(\'SKIPPING IMAGE\',member,str(e)[:160]);missing+=1\n    if not rows:raise RuntimeError(\'No test crops could be read, inspect archive structure/labels/password\')\n    y=[r[\'gt_trashnet\'] for r in rows];yhat=[r[\'prediction\'] for r in rows]\n    report=classification_report(y,yhat,labels=[\'cardboard\',\'metal\',\'plastic\'],output_dict=True,zero_division=0)\n    from scipy.stats import binomtest\n    successes=sum(a==b for a,b in zip(y,yhat))\n    ci=binomtest(successes,len(rows)).proportion_ci(confidence_level=.95,method=\'exact\')\n    result=dict(model=\'ResNet18 fine-tune on TrashNet\',dataset=\'ZeroWaste-f\',split=args.split,seed=args.seed,\n         labels_evaluated=[\'cardboard\',\'metal\',\'plastic\'],excluded_ZW_labels=\'none: soft/rigid plastic mapped to plastic\',\n         count=len(rows),missing_or_unreadable=missing,per_class_gt={s:y.count(s) for s in set(y)},\n         top1_accuracy=accuracy_score(y,yhat),macro_f1_three_labels=f1_score(y,yhat,labels=[\'cardboard\',\'metal\',\'plastic\'],average=\'macro\',zero_division=0),\n         accuracy_95pct_exact_CI=[ci.low,ci.high],note=\'External-domain CLASSIFICATION of GT bounding-box crops only. Not detector mAP, not video throughput, not representative of all ZeroWaste frames.\')\n    (dst/\'zerowaste_evaluation_metrics.json\').write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    (dst/\'classification_report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    with open(dst/\'zerowaste_predictions.csv\',\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)\n    (dst/\'READ_ME.txt\').write_text(\'ZeroWaste-f external-domain evaluation. Results use official COCO ground-truth crops from split \'+args.split+ \'. Keep images and source archive private; dataset licence CC BY-NC 4.0. Check validity of categories and sampling before citing results.\\n\',encoding=\'utf-8\')\n    print(\'DONE: ZeroWaste-f\',\'samples:\',len(rows),\'accuracy:\',round(result[\'top1_accuracy\']*100,2),\'macro F1:\',round(result[\'macro_f1_three_labels\']*100,2))\n    print(\'Saved results to\',dst.resolve());print(\'Sample count by material:\',result[\'per_class_gt\'])\n\nif __name__==\'__main__\':main()\n',encoding='utf-8')
import subprocess,sys
cmd=[sys.executable,str(SCRIPT),'--archive',str(ARCHIVE),'--checkpoint',CKPT,'--split','val','--per-class','25','--output','/content/zw_eval_outputs']
subprocess.run(cmd,check=True)

In [ ]:
import shutil, json
from google.colab import files
root='/content/zw_eval_outputs'
print(json.dumps(json.load(open(f'{root}/zerowaste_evaluation_metrics.json')),indent=2))
shutil.make_archive('/content/KAUST_ZeroWaste_External_Evaluation','zip',root)
files.download('/content/KAUST_ZeroWaste_External_Evaluation.zip')

### Reporting note

State the exact dataset split, sample size, selection rules, resizing (160×160 pixels), metrics and class mapping. **Never call this a detector or segmentation evaluation.** If the ZeroWaste-f download is not completed before submission, do not enter fabricated scores; list the domain-shift evaluation as pending and submit the verified Phase 1 work.